<div style="background:linear-gradient(135deg,#0f0c29,#302b63);padding:20px 24px;border-radius:10px;border-left:5px solid #38bdf8;font-family:Arial,sans-serif;">
  <h2 style="margin:0;color:#38bdf8;">🛒&nbsp;Amazon Alexa Reviews — NLP Analysis</h2>
  <p style="margin:8px 0 0;color:#bbb;font-size:14px;">EDA · TextBlob + BERT sentiment · TF-IDF + Logistic Regression · WordCloud · ratings over time</p>
</div>

## Overview

End-to-end NLP analysis of Amazon Alexa product reviews.

| Step | Method |
|------|--------|
| Sentiment labeling | Rating-based + TextBlob polarity |
| Text vectorization | TF-IDF |
| Classification | Logistic Regression |
| Deep sentiment | `nlptown/bert-base-multilingual-uncased-sentiment` |
| Visualization | WordCloud, seaborn, matplotlib |

**Dataset:** `amazon_alexa.tsv` — place in the same directory as this notebook.

```bash
pip install pandas numpy matplotlib seaborn wordcloud textblob scikit-learn transformers
```

## 1. Load & Clean Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("amazon_alexa.tsv", sep="\t")
df["date"] = pd.to_datetime(df["date"], format="mixed")
df = df.sort_values("date").reset_index(drop=True)

# Sentiment label from rating
df["sentiment"] = df["rating"].apply(lambda x: "Positive" if x >= 4 else "Negative")

# Clean text
df["clean_text"] = (
    df["verified_reviews"]
    .fillna("")
    .str.lower()
    .str.replace(r"[^a-z\s]", "", regex=True)
    .str.strip()
)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nSentiment counts:\n{df['sentiment'].value_counts()}")
df.head(3)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Amazon Alexa Reviews — Overview", fontsize=14, fontweight="bold")

# Rating distribution
sns.countplot(x="rating", data=df, hue="rating", palette="viridis", legend=False, ax=axes[0])
axes[0].set_title("Rating Distribution")
axes[0].set_xlabel("Stars")

# Sentiment split
df["sentiment"].value_counts().plot(kind="pie", autopct="%1.1f%%",
    colors=["#4ade80","#f87171"], startangle=90, ax=axes[1])
axes[1].set_title("Sentiment Split")
axes[1].set_ylabel("")

# Reviews over time
df.groupby(df["date"].dt.to_period("M")).size().plot(ax=axes[2], color="#38bdf8")
axes[2].set_title("Reviews Over Time")
axes[2].set_xlabel("")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 3. WordCloud — Positive vs Negative

In [ ]:
from wordcloud import WordCloud

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, cmap, title in [
    (ax1, "Positive", "Greens",  "Positive Reviews"),
    (ax2, "Negative", "Reds",    "Negative Reviews"),
]:
    text = " ".join(df[df["sentiment"] == label]["clean_text"])
    wc = WordCloud(width=600, height=300, background_color="white",
                   colormap=cmap, max_words=80).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.axis("off")

plt.suptitle("Most Common Words by Sentiment", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Top Words — Positive vs Negative

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, color, title in [
    (ax1, "Positive", "#4ade80", "Top 15 Positive Words"),
    (ax2, "Negative", "#f87171", "Top 15 Negative Words"),
]:
    words = " ".join(df[df["sentiment"] == label]["clean_text"]).split()
    top = Counter(words).most_common(15)
    words_list, counts = zip(*top)
    ax.barh(words_list[::-1], counts[::-1], color=color, alpha=0.8)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Frequency")

plt.tight_layout()
plt.show()

## 5. ML Model — TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["clean_text"])
y = df["sentiment"].map({"Positive": 1, "Negative": 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=["Negative","Positive"]))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    display_labels=["Negative","Positive"], colorbar=False, ax=ax)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 6. BERT Sentiment — Deep Analysis on Sample Reviews

> Uses `nlptown/bert-base-multilingual-uncased-sentiment`. First run downloads ~600 MB. GPU recommended but CPU works.

In [ ]:
from transformers import pipeline

bert = pipeline("sentiment-analysis",
                model="nlptown/bert-base-multilingual-uncased-sentiment",
                device=-1)  # -1 = CPU

samples = df["verified_reviews"].dropna().sample(10, random_state=42).tolist()

results = []
for review in samples:
    r = bert(review[:512])[0]  # truncate to 512 tokens
    stars = int(r["label"][0])
    results.append({"review": review[:80] + "...", "stars": stars, "score": round(r["score"], 3)})

pd.DataFrame(results).sort_values("stars", ascending=False)